# Week 2 · The line dashboard: find the bottleneck

**Sentence of the day:** A process can only go as fast as its slowest step, and that step sets the output for everyone.

**What you will build today:** a small model of a production line and, at the end, a **dashboard**: one picture with three key numbers at the top and two charts below, with sliders you can move. When you add an oven or a worker, the red bar shows you where the bottleneck has moved.

**How this notebook works**

- Google Colab is a notebook: a document where text and runnable code sit side by side. You run a code cell with the ▶ button on its left, or with **Shift + Enter**.
- Run the cells **from top to bottom**. A cell can only use what the cells above it have created.
- **Part 1** is complete: we run it together (10 minutes). **Part 2** and **Part 3** are for your team. Cells marked **✏️ YOUR TURN** contain `...` or a `# TODO`: replace them with your code.
- Every step ends with a **check cell**. It prints ✅ when your result matches the hand calculation, and ❌ with a hint when it does not.
- You may use an AI assistant. Check its numbers with the check cells and by hand.

| Part | What | Time |
|---|---|---|
| 0 | Setup | 1 min |
| 1 | Worked set-up: the party pizza line (complete) | 10 min, together |
| 2 | Practice: Mensa, Backhaus Kaya, what-if, crossover | 15 min |
| 3 | Build your dashboard in four small steps | 15 min |
| Extra | Optional extensions for quick teams | if time |

## Part 0 · Setup

This cell loads four **libraries**. A library is a toolbox other people have written, so we do not have to: `pandas` for tables, `numpy` for numbers, `matplotlib` for charts, `ipywidgets` for sliders. All four are already installed in Colab.

The cell also sets the colours we use all day (grey bars, **red for the bottleneck**) and defines `check`, the helper that prints ✅ or ❌. Run it once.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets

# Our colours: grey for normal steps, red for the bottleneck, dark grey for text
GREY, RED, DARK, LIGHT = "#9e9e9e", "#c1002a", "#1a1a1a", "#f2f2f2"
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "text.color": DARK, "axes.labelcolor": DARK, "xtick.color": DARK, "ytick.color": DARK})

def check(label, got, expected, tol=0.01, hint=""):
    """Compare your result with the hand calculation and print ✅ or ❌."""
    try:
        ok = abs(float(got) - float(expected)) <= tol
    except Exception:
        ok = got == expected
    print(("✅ " if ok else "❌ ") + f"{label}: you have {got}, expected {expected}" + ("" if ok else f"   Hint: {hint}"))

print("Setup done.")

## Part 1 · Worked set-up: the party pizza line

Four friends make pizzas for last week's party: **24 pizzas** are needed. Every pizza goes through four steps, one after the other. This is a **line** (a process where every unit goes through the same steps in the same order).

### 1a · The line as a table

We store the line as a **DataFrame**: a table in Python, with named columns, like a small spreadsheet. One row per step.

- `minutes_per_unit`: minutes for one pass through the step. For most steps one pass = one pizza. For the oven one pass = one **load**.
- `parallel`: how many people or machines do this step at the same time.
- `units_per_load`: how many pizzas one pass makes (1 for a person, 2 for the oven, which holds two pizzas).

In [ ]:
pizza = pd.DataFrame({
    "step":             ["Roll dough", "Add toppings", "Bake", "Cut & serve"],
    "minutes_per_unit": [3,            4,              10,     2],      # minutes for one pass
    "parallel":         [1,            1,              1,      1],      # people or ovens working side by side
    "units_per_load":   [1,            1,              2,      1],      # the oven bakes 2 pizzas per load
}).set_index("step")     # use the step name as the row label

pizza

### 1b · Capacity of each step

**Capacity of a step** (units per hour) = 60 minutes ÷ minutes per unit × number of parallel workers or machines. For a batch machine such as the oven: units per load × 60 ÷ minutes per load.

Our formula below does both at once (for a person, units per load is simply 1).

A **function** is a reusable recipe with a name: you give it ingredients (here: a table) and it gives something back. `capacity_per_hour(df)` adds a new column `capacity_per_hour` to the table. (`df` is short for DataFrame: whatever table we hand to the recipe.)

In [ ]:
def capacity_per_hour(df):
    """Add a column: how many units each step can make per hour."""
    df["capacity_per_hour"] = (60 / df["minutes_per_unit"] * df["parallel"] * df["units_per_load"]).round(1)
    return df

capacity_per_hour(pizza)

### 1c · Bottleneck, line output, throughput time

- **Bottleneck** = the step with the lowest capacity. **Line capacity** (line output) = capacity of the bottleneck.
- **Bottleneck time** = minutes per unit at the bottleneck: after the first pizza, one pizza comes out every bottleneck time (60 ÷ 12 = 5 minutes).
- **Throughput time** = time one unit needs from first to last step without waiting = sum of the step times.
- Time for *n* pizzas = throughput time of the first pizza + (n − 1) × bottleneck time.

`idxmin()` gives the row label of the smallest value in a column: exactly "the step with the lowest capacity".

In [ ]:
def bottleneck(df):
    """Name of the step with the lowest capacity."""
    return df["capacity_per_hour"].idxmin()

def line_output(df):
    """Units per hour the whole line can make = capacity of the bottleneck."""
    return df["capacity_per_hour"].min()

def throughput_time(df):
    """Minutes one unit needs from the first to the last step, without waiting."""
    return df["minutes_per_unit"].sum()

def time_for(df, n):
    """Minutes to make n units: the first one takes the throughput time, then one every bottleneck time."""
    bottleneck_time = 60 / line_output(df)
    return throughput_time(df) + (n - 1) * bottleneck_time

print("Bottleneck:", bottleneck(pizza))
print("Line output:", line_output(pizza), "pizzas per hour")
print("Throughput time:", throughput_time(pizza), "minutes for one pizza")
print("24 pizzas take:", time_for(pizza, 24), "minutes")

### 1d · Utilization of each step

**Utilization** = actual output ÷ design capacity, in %. If the line makes 12 pizzas per hour, every step makes 12 pizzas per hour, whatever its capacity. So the dough roller (capacity 20) is busy 12 ÷ 20 = 60 % of the time and idle the rest.

In [ ]:
def utilization(df, output):
    """Add a column: how busy each step is (in %) when the line makes `output` units per hour."""
    df["utilization_pct"] = (output / df["capacity_per_hour"] * 100).round(1)
    return df

utilization(pizza, line_output(pizza))

In [ ]:
# CHECK Part 1
try:
    check("capacities (dough, toppings, bake, cut)", list(pizza["capacity_per_hour"]), [20, 15, 12, 30])
    check("bottleneck", bottleneck(pizza), "Bake")
    check("line output (pizzas per hour)", line_output(pizza), 12)
    check("throughput time (minutes)", throughput_time(pizza), 19, hint="3 + 4 + 10 + 2")
    check("24 pizzas (minutes)", time_for(pizza, 24), 134, hint="19 + 23 × 5")
    check("utilization % (dough, toppings, bake, cut)", list(pizza["utilization_pct"]), [60, 80, 100, 40])
except Exception as e:
    print("❌ Not working yet:", e)

### 1e · Two charts that tell the story

We write each chart as a function, so we can reuse it later in the dashboard. `ax` is one drawing area inside a figure (a figure can hold several).

- **Capacity chart:** every step's capacity, the bottleneck in red, the demand as a dashed line. If the red bar is below the dashed line, the line cannot keep up.
- **Utilization chart:** how busy each step is. Only the bottleneck reaches 100 %.

A good chart title states the message, not just the topic.

In [ ]:
def draw_capacity(ax, df, demand, unit="pizzas"):
    """Bar chart of capacity per hour; bottleneck in red; dashed line = demand."""
    colours = [RED if step == bottleneck(df) else GREY for step in df.index]
    bars = ax.bar(df.index, df["capacity_per_hour"], color=colours, width=0.6)
    ax.bar_label(bars, labels=[f"{v:,.0f}" for v in df["capacity_per_hour"]], padding=3)
    ax.axhline(demand, color=DARK, linestyle="--", linewidth=1.2)
    ax.set_ylim(0, max(df["capacity_per_hour"].max(), demand) * 1.18)
    ax.set_title(f"Capacity per hour ({unit}) · dashed line = demand {demand:,.0f}", loc="left", fontsize=11)

def draw_utilization(ax, df):
    """Bar chart of utilization in %; bottleneck in red."""
    colours = [RED if step == bottleneck(df) else GREY for step in df.index]
    bars = ax.bar(df.index, df["utilization_pct"], color=colours, width=0.6)
    ax.bar_label(bars, labels=[f"{v:.0f} %" for v in df["utilization_pct"]], padding=3)
    ax.axhline(100, color=DARK, linestyle=":", linewidth=1)
    ax.set_ylim(0, 118)
    ax.set_title("Utilization: how busy each step is (%)", loc="left", fontsize=11)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
draw_capacity(left, pizza, demand=12)          # 24 pizzas in 2 hours = 12 per hour
draw_utilization(right, pizza)
fig.suptitle(f"{bottleneck(pizza)} sets the pace: {line_output(pizza):.0f} pizzas per hour",
             x=0.01, ha="left", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Look at the charts:** a second person on the toppings would make that grey bar taller. Would the party get its pizzas sooner? (You will test this in P3 and in the dashboard.)

## Part 2 · Practice

Now you use the same functions for three new lines. Each problem: fill in the `...`, run the cell, run the check.

### P1 · The Mensa puzzle

At the Mensa at 12:15 the queue reaches the door, yet the cashier is idle half the time. Why?

Per guest: main-dish counter 30 seconds (1 server), cashier 15 seconds (1 cashier). (Taking a tray takes 10 seconds, but everyone serves themselves, so it is never the constraint: we leave it out.) At 12:15, **180 guests per hour** arrive (3 per minute).

**✏️ YOUR TURN:** our table counts in minutes, so first turn seconds into minutes (30 s = ? min).

In [ ]:
mensa = pd.DataFrame({
    "step":             ["Main-dish counter", "Cashier"],
    "minutes_per_unit": [..., ...],          # TODO: 30 seconds and 15 seconds, written in minutes
    "parallel":         [1, 1],
    "units_per_load":   [1, 1],
}).set_index("step")
ARRIVALS = 180                               # guests per hour at 12:15

capacity_per_hour(mensa)
mensa_output = ...                           # TODO: guests served per hour = capacity of the bottleneck
queue_growth = ...                           # TODO: guests who arrive per hour − guests served per hour
utilization(mensa, mensa_output)

print(f"The queue grows by {queue_growth:.0f} guests per hour.")
mensa

In [ ]:
# CHECK P1
try:
    check("capacities (counter, cashier)", list(mensa["capacity_per_hour"]), [120, 240], hint="60 ÷ 0.5 and 60 ÷ 0.25")
    check("bottleneck", bottleneck(mensa), "Main-dish counter")
    check("queue growth (guests per hour)", queue_growth, 60, hint="180 − 120")
    check("cashier utilization %", mensa.loc["Cashier", "utilization_pct"], 50, hint="120 ÷ 240")
except Exception as e:
    print("❌ Not working yet:", e)

**Think:** the Mensa manager wants to open a second cash desk. What happens to the queue? And with a second server at the counter instead? Change one number in a copy of the table and find out:

In [ ]:
two_servers = mensa.copy()                       # a copy, so the original table stays as it is
two_servers.loc["Main-dish counter", "parallel"] = 2
capacity_per_hour(two_servers)
print("With a second server:", line_output(two_servers), "guests per hour; the queue grows by",
      ARRIVALS - line_output(two_servers), "per hour (negative = it shrinks).")
check("line output with a second server", line_output(two_servers), 240, hint="the counter now makes 240, the cashier too: both are bottlenecks")

### P2 · Backhaus Kaya: the roll line in the night shift

Aylin Kaya must answer FrischMarkt by Monday: **1,800 extra rolls per night**. The rolls are baked in a **4-hour window** (02:00–06:00).

| Step | Resource | One pass |
|---|---|---|
| Mix dough | 1 spiral mixer | 600 rolls of dough per batch, 20 min |
| Divide & shape | 2 bakers | 7 rolls per minute each |
| Proof | 1 proofing cabinet | 720 rolls, 45 min |
| Bake | 1 deck oven | 240 rolls per load, 20 min (18 bake + 2 load/unload) |
| Cool & pack | 1 person | 15 rolls per minute |

Three capacity words:
- **Design capacity** = the most the process could make in ideal conditions.
- **Effective capacity** = what it can make with planned losses (set-ups, cleaning, breaks, product mix): here **600 rolls per hour**.
- **Actual output** = what it really made: last week **540 rolls per hour**.

**Utilization** = actual output ÷ design capacity. **Efficiency** = actual output ÷ effective capacity.

**✏️ YOUR TURN:** fill in the missing numbers. Tip: "7 rolls per minute" = one pass of 1 minute that makes 7 rolls.

In [ ]:
kaya = pd.DataFrame({
    "step":             ["Mix dough", "Divide & shape", "Proof", "Bake", "Cool & pack"],
    "minutes_per_unit": [20,          1,                45,      ...,    1],       # TODO: minutes per oven load
    "parallel":         [1,           ...,              1,       1,      1],       # TODO: how many bakers shape?
    "units_per_load":   [600,         7,                720,     ...,    15],      # TODO: rolls per oven load
}).set_index("step")
capacity_per_hour(kaya)

WINDOW = 4                    # hours of baking per night (02:00–06:00)
EFFECTIVE, ACTUAL = 600, 540  # rolls per hour
OWN_SHOP, CONTRACT = 1500, 1800   # rolls per night

design_per_night = ...                        # TODO: line output × hours in the window
kaya_utilization = ...                        # TODO: actual ÷ design capacity × 100
kaya_efficiency = ...                         # TODO: actual ÷ effective capacity × 100
gap = (OWN_SHOP + CONTRACT) - EFFECTIVE * WINDOW   # rolls per night we cannot make

print(f"Bottleneck: {bottleneck(kaya)}, {line_output(kaya):,.0f} rolls per hour, {design_per_night:,.0f} per night")
print(f"Utilization {kaya_utilization:.1f} %, efficiency {kaya_efficiency:.1f} %, gap {gap:,.0f} rolls per night")
kaya

In [ ]:
# CHECK P2
try:
    check("capacities (mix, shape, proof, bake, pack)", list(kaya["capacity_per_hour"]), [1800, 840, 960, 720, 900])
    check("bottleneck", bottleneck(kaya), "Bake", hint="the deck oven")
    check("design capacity per night", design_per_night, 2880, hint="720 × 4")
    check("utilization %", round(kaya_utilization, 1), 75.0, hint="540 ÷ 720")
    check("efficiency %", round(kaya_efficiency, 1), 90.0, hint="540 ÷ 600")
    check("gap (rolls per night)", gap, 900, hint="1,500 + 1,800 − 600 × 4")
except Exception as e:
    print("❌ Not working yet:", e)

### P3 · What if? Jonas wants a third shaper

Head baker Jonas Weber says: *"We just hire a third shaper."* Test his idea, then test a second deck oven. Always change a **copy** of the table (`.copy()`), so `kaya` stays as it is.

**✏️ YOUR TURN:** complete the two TODO lines.

In [ ]:
third_shaper = kaya.copy()
third_shaper.loc["Divide & shape", "parallel"] = 3
capacity_per_hour(third_shaper)

second_oven = kaya.copy()
...                                    # TODO: set the number of ovens (parallel at "Bake") to 2
capacity_per_hour(second_oven)

print("Third shaper:", line_output(third_shaper), "rolls per hour, bottleneck:", bottleneck(third_shaper))
print("Second oven: ", line_output(second_oven), "rolls per hour, bottleneck:", bottleneck(second_oven))

# The same idea for the party: the neighbour lends a second oven
two_ovens = pizza.copy()
two_ovens.loc["Bake", "parallel"] = 2
capacity_per_hour(two_ovens)
pizzas_two_ovens = ...                 # TODO: minutes for 24 pizzas with two ovens (use time_for)
print("Party with two ovens: bottleneck", bottleneck(two_ovens), "→ 24 pizzas in", pizzas_two_ovens, "minutes")

In [ ]:
# CHECK P3
try:
    check("third shaper: line output", line_output(third_shaper), 720, hint="shaping was not the bottleneck: nothing changes")
    check("second oven: new bottleneck", bottleneck(second_oven), "Divide & shape", hint="oven 1,440 now; which step is lowest?")
    check("second oven: line output", line_output(second_oven), 840)
    check("party, two ovens: bottleneck", bottleneck(two_ovens), "Add toppings")
    check("party, two ovens: 24 pizzas (minutes)", pizzas_two_ovens, 111, hint="19 + 23 × 4")
except Exception as e:
    print("❌ Not working yet:", e)

**Lesson:** the bottleneck moves. Extra people at a step that is *not* the bottleneck change nothing.

### P4 · Crossover: current process or a semi-automatic roll line?

Option B in the case: a semi-automatic roll line costs **36,000 € per year** (lease and maintenance), but each roll costs only **0.10 €** to make instead of **0.18 €** today. Which process is cheaper depends on the volume.

- Total cost = fixed cost + variable cost per unit × volume.
- **Crossover volume** of two processes = (fixed cost B − fixed cost A) ÷ (variable cost A − variable cost B).
- Contract volume = 1,800 rolls × 6 days × 50 weeks. Price: 0.32 € per roll.

**✏️ YOUR TURN:** complete the three TODO lines. The chart below is given.

In [ ]:
import math
FIXED_A, VAR_A = 0, 0.18          # current process: no extra fixed cost
FIXED_B, VAR_B = 36000, 0.10      # semi-automatic roll line
PRICE = 0.32
contract_volume = 1800 * 6 * 50   # rolls per year

crossover = ...                                         # TODO: (fixed B − fixed A) ÷ (variable A − variable B)
contribution_A = (PRICE - VAR_A) * contract_volume - FIXED_A
contribution_B = ...                                    # TODO: the same for the roll line
break_even_B = math.ceil(...)                           # TODO: fixed B ÷ (price − variable B), rounded up

print(f"Crossover: {crossover:,.0f} rolls per year; contract: {contract_volume:,.0f} rolls per year")
print(f"Contribution per year: current {contribution_A:,.0f} €, roll line {contribution_B:,.0f} €")

In [ ]:
# CHECK P4
try:
    check("crossover volume (rolls per year)", round(crossover), 450000, hint="36,000 ÷ (0.18 − 0.10)")
    check("contract volume (rolls per year)", contract_volume, 540000)
    check("contribution current process (€)", round(contribution_A), 75600, hint="(0.32 − 0.18) × 540,000")
    check("contribution roll line (€)", round(contribution_B), 82800, hint="(0.32 − 0.10) × 540,000 − 36,000")
    check("break-even of the roll line (rolls)", break_even_B, 163637, hint="36,000 ÷ 0.22, rounded UP")
except Exception as e:
    print("❌ Not working yet:", e)

In [ ]:
# The cost-vs-volume chart (given): where do the two lines cross?
volume = np.linspace(0, 800000, 81)                     # 0 to 800,000 rolls per year
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(volume, FIXED_A + VAR_A * volume, color=GREY, linewidth=2.5, label="current process (0.18 € per roll)")
ax.plot(volume, FIXED_B + VAR_B * volume, color=DARK, linewidth=2.5, label="roll line (36,000 € + 0.10 € per roll)")
x = crossover
ax.plot(x, FIXED_B + VAR_B * x, "o", color=RED, markersize=9)
ax.annotate(f"crossover {x:,.0f}", (x, FIXED_B + VAR_B * x), xytext=(-150, 25), textcoords="offset points",
            color=RED, fontweight="bold", arrowprops=dict(arrowstyle="-", color=RED))
ax.axvline(contract_volume, color=RED, linestyle="--", linewidth=1.2)
ax.text(contract_volume, 10000, f"  contract {contract_volume:,.0f}", color=RED)
ax.xaxis.set_major_formatter(lambda v, pos: f"{v/1000:,.0f}k")
ax.yaxis.set_major_formatter(lambda v, pos: f"{v/1000:,.0f}k €")
ax.set_xlabel("rolls per year"); ax.set_ylabel("total cost per year")
ax.set_title(f"Above {crossover:,.0f} rolls per year the roll line is cheaper; the contract brings {contract_volume:,.0f}",
             loc="left", fontsize=12, fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

**Think:** the FrischMarkt contract can be cancelled after one year, but the lease runs three years. Without the contract, Backhaus Kaya's extra volume is zero. Which line in the chart would Aylin then be on?

## Part 3 · Build your dashboard in four small steps

A **dashboard** is one picture that shows the few numbers a manager needs, at a glance. Ours has three **tiles** at the top (big number + small label; the bottleneck tile is red) and the two charts from Part 1 below. Then we add sliders.

| Step | What you build |
|---|---|
| D1 | `kpis(df, demand)`: the four key numbers in one go |
| D2 | `draw_dashboard(df, demand)`: tiles + two charts in one figure |
| D3 | sliders: change the line and watch the red bar move |
| D4 | optional: compare two scenarios side by side |

First, the Backhaus Kaya roll line once more (complete), so that Part 3 also works if your team has not finished Part 2.

In [ ]:
rolls = pd.DataFrame({
    "step":             ["Mix dough", "Divide & shape", "Proof", "Bake", "Cool & pack"],
    "minutes_per_unit": [20,          1,                45,      20,     1],
    "parallel":         [1,           2,                1,       1,      1],
    "units_per_load":   [600,         7,                720,     240,    15],
}).set_index("step")
ROLL_DEMAND = (1500 + 1800) / 4      # own shop + contract, per hour of the 4-hour window = 825
capacity_per_hour(rolls)

### D1 · The key numbers

KPI means *key performance indicator*: one of the few numbers that tell you how the process is doing. Our function returns four of them in a **dictionary** (a list of name → value pairs, written `{"name": value}`).

One new idea: the line makes what is demanded, **but never more than the bottleneck allows**. So the real output is the smaller of demand and line output: `min(demand, line_output(df))`. If demand is lower than capacity, even the bottleneck has idle time.

**✏️ YOUR TURN:** complete the two TODO lines.

In [ ]:
def kpis(df, demand):
    """The four key numbers of a line for a given demand (units per hour)."""
    capacity_per_hour(df)                          # make sure the capacity column is up to date
    output = min(demand, line_output(df))          # the line makes what is demanded, but never more than it can
    b = bottleneck(df)
    return {
        "line_output": ...,                                                    # TODO: units per hour the line can make
        "bottleneck": b,
        "bottleneck_utilization": output / df.loc[b, "capacity_per_hour"] * 100,
        "throughput_time": ...,                                                # TODO: minutes for one unit
    }

kpis(pizza, demand=12)

In [ ]:
# CHECK D1
try:
    k = kpis(pizza, demand=12)
    check("line output", k["line_output"], 12)
    check("bottleneck", k["bottleneck"], "Bake")
    check("bottleneck utilization at demand 12 (%)", k["bottleneck_utilization"], 100)
    check("throughput time (min)", k["throughput_time"], 19)
    check("bottleneck utilization at demand 9 (%)", kpis(pizza, demand=9)["bottleneck_utilization"], 75,
          hint="9 pizzas demanded, the oven could make 12")
    check("roll line output", kpis(rolls, ROLL_DEMAND)["line_output"], 720)
except Exception as e:
    print("❌ Not working yet:", e)

### D2 · Draw the dashboard

Three new tools:

- `plt.subplot_mosaic` builds a figure from a little text map of the layout: each name is one drawing area. Our map has the three tiles on top and the two charts below.
- `draw_tile` draws one tile: a grey box without axes, a big number and a small label, both written with `ax.text`. Positions go from 0 (left/bottom) to 1 (right/top) inside the box.
- An **f-string** (`f"..."`) puts numbers into text: `f"{12:.0f} pizzas"` gives `12 pizzas`.

**✏️ YOUR TURN:** fill in the two TODO lines: the big text of the bottleneck tile and of the throughput tile.

In [ ]:
def draw_tile(ax, big, small, colour=LIGHT, text_colour=DARK):
    """One KPI tile: coloured box, big number in the middle, small label below."""
    ax.set_facecolor(colour)
    ax.set_xticks([]); ax.set_yticks([])                  # no axis numbers
    for side in ax.spines.values():
        side.set_visible(False)                           # no frame
    ax.text(0.5, 0.58, big, ha="center", va="center", fontsize=26, fontweight="bold", color=text_colour)
    ax.text(0.5, 0.18, small, ha="center", va="center", fontsize=10, color=text_colour)

def draw_dashboard(df, demand, unit="pizzas"):
    """One figure: three KPI tiles on top, capacity and utilization charts below."""
    k = kpis(df, demand)
    utilization(df, min(demand, k["line_output"]))        # how busy each step is at the real output
    layout = [["out", "out", "bot", "bot", "time", "time"],
              ["cap", "cap", "cap", "use", "use", "use"]]
    fig, ax = plt.subplot_mosaic(layout, figsize=(11, 6.4), height_ratios=[1, 2.6])

    # Row 1: the three tiles
    draw_tile(ax["out"], f"{k['line_output']:,.0f}", f"{unit} per hour the line can make")
    draw_tile(ax["bot"], ...,                                       # TODO: the bottleneck's name
              f"bottleneck · {k['bottleneck_utilization']:.0f} % busy", colour=RED, text_colour="white")
    draw_tile(ax["time"], ...,                                      # TODO: throughput time, e.g. "19 min"
              f"throughput time of one {unit[:-1]}")

    # Row 2: the two charts from Part 1
    draw_capacity(ax["cap"], df, demand, unit)
    draw_utilization(ax["use"], df)

    # The title states the message
    short = demand - k["line_output"]                       # how many units per hour the line cannot make
    met = "demand is met" if short <= 0 else f"{short:,.0f} short of demand"
    fig.suptitle(f"{k['bottleneck']} sets the pace: {k['line_output']:,.0f} {unit} per hour · {met}",
                 x=0.01, ha="left", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

draw_dashboard(pizza, demand=12)

In [ ]:
# CHECK D2: the same function must work for a different line (the bakery)
try:
    draw_dashboard(rolls, ROLL_DEMAND, unit="rolls")
    print("✅ Dashboard drawn for the roll line too: is the red bar the oven? Does the title say 105 short?")
except Exception as e:
    print("❌ Not working yet:", e)

### D3 · Make it move: sliders

`interact` (from `ipywidgets`) turns the inputs of a function into **widgets**: sliders and drop-down menus. Each time you let go of a slider, the function runs again and the dashboard is redrawn.

Below, every step of the pizza line gets a slider for its minutes and one for its people (or ovens), plus one for demand. The function copies the pizza table, puts the slider values into it and draws the dashboard.

**✏️ YOUR TURN:** the oven slider is missing. Wire it in three small edits (look for TODO): add `ovens=slider(1, 3, 1)` to the `@interact(...)` list, add `ovens` to the function inputs, and delete the line `ovens = 1`. Then move it to 2: where does the red bar go?

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown

def slider(low, high, start):
    """A whole-number slider that redraws when you let go of it (keeps Colab fast)."""
    return IntSlider(min=low, max=high, value=start, continuous_update=False)

@interact(dough_min=slider(1, 8, 3), topping_min=slider(1, 8, 4), bake_min=slider(4, 16, 10), cut_min=slider(1, 8, 2),
          dough_people=slider(1, 3, 1), topping_people=slider(1, 3, 1),    # TODO: add ovens=slider(1, 3, 1) here
          cut_people=slider(1, 3, 1), demand=slider(4, 30, 12))
def play_pizza(dough_min, topping_min, bake_min, cut_min,
               dough_people, topping_people, cut_people, demand):          # TODO: add ovens after topping_people
    ovens = 1                                                              # TODO: delete this line
    line = pizza.copy()
    line["minutes_per_unit"] = [dough_min, topping_min, bake_min, cut_min]   # one value per step, in step order
    line["parallel"] = [dough_people, topping_people, ovens, cut_people]
    draw_dashboard(line, demand, unit="pizzas")

**Now switch lines.** The drop-down menu chooses the line; the slider adds people or machines at today's bottleneck. Move it from 0 to 1: the red bar jumps to a different step. That is the sentence of the day in one picture.

In [ ]:
LINES = {"Party pizza": (pizza, 12, "pizzas"),                  # table, demand per hour, unit
         "Backhaus Kaya rolls": (rolls, ROLL_DEMAND, "rolls")}

@interact(line=Dropdown(options=list(LINES)), extra_at_bottleneck=slider(0, 2, 0))
def play_line(line, extra_at_bottleneck):
    table, demand, unit = LINES[line]
    df = table.copy()
    b = bottleneck(capacity_per_hour(df))                        # today's bottleneck
    df.loc[b, "parallel"] += extra_at_bottleneck                 # add people or machines there
    draw_dashboard(df, demand, unit)

**Questions for your team** (write the answers here)

1. Pizza line: with two ovens, which step is the bottleneck, and how busy is the oven now?
2. Roll line: with one extra oven the line makes 840 rolls per hour. Is the demand of 825 per hour met? What is the next step to add capacity at?
3. Pizza line: set demand to 8. Why is the oven tile no longer at 100 %?

*Your answers:*

1.
2.
3. 

### D4 · Optional: two scenarios side by side

Managers like to compare **before** and **after** in one picture. Ideas, pick one:

- **Scenario toggle:** draw the capacity chart of "today" on the left and of a changed line on the right (`plt.subplots(1, 2)`), with buttons to choose the change (`ToggleButtons` from `ipywidgets`).
- **Colour by utilization:** instead of grey/red, colour every bar by how busy the step is (light = idle, dark red = 100 %). Hint: `plt.cm.Reds(u / 100)` gives a colour for a utilization `u`.

In [ ]:
# ✏️ D4 · your scenario comparison here (optional)
# 1. make a changed copy of pizza (or rolls)
# 2. fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
# 3. draw_capacity(left, ...) and draw_capacity(right, ...), give each a title with its line output
# 4. plt.show()


## Optional extensions for quick teams

### E1 · Real life is not exact: random step times

Our model says one pizza every 5 minutes. But nobody rolls dough in exactly 3 minutes: sometimes 2.5, sometimes 3.5. Here each step time is drawn at random, up to 20 % faster or slower (a *triangular* distribution: most often the planned time, rarely the extremes). The `seed` makes the draw repeatable, so everyone gets the same picture.

The simulation sends 50 pizzas through the line, one after the other. A pizza can start a step only when it has left the step before **and** a worker (or the oven) is free. The oven starts a load when both pizzas are ready. This is a preview of next week's topic: **variability**.

In [ ]:
def simulate(df, n=50, spread=0.2, seed=1):
    """Minute at which each of n units leaves the line, with random step times (± spread)."""
    rng = np.random.default_rng(seed)
    ready = np.zeros(n)                                    # minute at which each unit is ready for the next step
    for step, s in df.iterrows():
        free = np.zeros(int(s["parallel"]))                # minute at which each worker or machine is free again
        size = int(s["units_per_load"])
        for first in range(0, n, size):                    # one pass at a time (1 pizza, or 2 in the oven)
            units = slice(first, first + size)
            m = s["minutes_per_unit"]
            minutes = rng.triangular((1 - spread) * m, m, (1 + spread) * m)
            w = free.argmin()                              # the worker who is free first
            start = max(ready[units].max(), free[w])       # wait for the units AND the worker
            free[w] = start + minutes
            ready[units] = free[w]
    return ready

finish = np.sort(simulate(pizza))
plan = throughput_time(pizza) + np.arange(50) * 60 / line_output(pizza)     # 19 min, then one every 5 min

fig, ax = plt.subplots(figsize=(9, 4))
ax.step(plan, np.arange(1, 51), where="post", color=GREY, linewidth=2, label="plan: 12 pizzas per hour")
ax.step(finish, np.arange(1, 51), where="post", color=RED, linewidth=2, label="simulation: random step times")
ax.set_xlabel("minutes"); ax.set_ylabel("pizzas finished")
ax.set_title(f"With random step times, 50 pizzas take {finish[-1]:.0f} min instead of {plan[-1]:.0f}: "
             f"{50 / finish[-1] * 60:.1f} per hour", loc="left", fontsize=12, fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

Change `seed=1` to other numbers (in `simulate(pizza, seed=...)`). Is the simulation ever faster than the plan? Why do the ups and downs not cancel out? (Hint: when the toppings step is slow, the oven waits, and that lost oven time never comes back.)

### E2 · Ask an AI to add a fourth tile, then check it

Ask an AI assistant: *"Add a fourth KPI tile to my `draw_dashboard` function that shows the bottleneck time in minutes per unit."* Paste its code into the cell below and run it for the pizza line. Then check the number by hand (60 ÷ 12 = ?).

In [ ]:
# E2 · paste the AI's version of draw_dashboard here and run it
# draw_dashboard(pizza, demand=12)


**AI reflection** (two sentences)

- What did the AI change in the function?
- How did you check its number, and was it right?

*Your answers:* 

## Three things to remember

1. **Capacity of a step** = 60 ÷ minutes per unit × parallel workers or machines. The **bottleneck** is the step with the lowest capacity, and it sets the output of the whole line.
2. Adding people or machines at a step that is **not** the bottleneck changes nothing. Add at the bottleneck, and the bottleneck **moves** to the next-slowest step.
3. **Utilization** = actual output ÷ design capacity; **efficiency** = actual output ÷ effective capacity. Only the bottleneck runs at 100 %; the others must have idle time.

> **Sentence of the day:** A process can only go as fast as its slowest step, and that step sets the output for everyone.

---
*Sources:* Party pizza line, Mensa puzzle and Backhaus Kaya: Eigene Darstellung, R. Bartnik (2026). Concepts: Heizer, J., Render, B., & Munson, C. (2023). *Operations management: Sustainability and supply chain management* (14th ed., Global ed.). Pearson, Chapter 7 (Process strategy) and Supplement 7 (Capacity and constraint management).

*Transparency note:* Claude (Anthropic) was used to draft and check this material; all content was reviewed by and is the responsibility of Roman Bartnik.